## Résumé exécutif

Une équipe des opérations logistiques prévoit un essai de terrain randomisé comparant trois stratégies de routage des chauffeurs (référence statique, réacheminement dynamique, optimisé par IA) sur trois régions de dépôt, avec le retard moyen de livraison (en minutes) comme réponse. PROC GLMPOWER prend un jeu de données *exemplaire* de moyennes de cellules conjecturées et résout le nombre total de chauffeurs nécessaires pour détecter les effets de stratégie à 80 % et 90 % de puissance, puis cartographie comment la puissance atteignable s'érode à mesure que la variabilité route-à-route augmente.

# Dimensionner un essai de terrain de routage des chauffeurs avec PROC GLMPOWER

## Résumé exécutif

Une équipe des opérations logistiques s'apprête à lancer un essai de terrain randomisé comparant trois stratégies de routage des chauffeurs -- une référence **Static**, un système de réacheminement **Dynamic**, et un planificateur **AI-Optimized** -- déployées sur trois régions de dépôt (Northeast, Midwest, West). La réponse est le **retard de livraison moyen en minutes**. Avant d'engager la capacité de la flotte dans l'essai, l'équipe doit répondre à : *combien de chauffeurs nous faut-il pour détecter de façon fiable l'amélioration opérationnelle attendue ?*

Ce notebook utilise **PROC GLMPOWER** pour effectuer une analyse prospective de puissance et de taille d'échantillon pour le modèle linéaire général sous-jacent à l'essai. En partant d'un jeu de données *exemplaire* de moyennes de cellules conjecturées, il (1) résout l'effectif total nécessaire pour atteindre 80 % et 90 % de puissance pour les effets globaux de stratégie et de région, (2) cartographie comment la puissance atteignable se dégrade à mesure que la variabilité route-à-route augmente, et (3) produit une courbe de puissance pour appuyer la décision d'effectif.

> **Point clé à retenir :** Sous un écart-type d'erreur plausible de 8 minutes, environ **63 chauffeurs** délivrent 80 % de puissance et **83 chauffeurs** délivrent 90 % de puissance pour détecter les effets de stratégie de routage -- mais si la variabilité du retard se rapproche de 10 minutes, même 90 chauffeurs inscrits n'atteignent pas 70 % de puissance, ce qui souligne la valeur de routes précises et bien instrumentées.

---
## 1. Construire le plan exemplaire

La première étape encode la structure de l'essai et le retard moyen *conjecturé* par l'équipe pour chaque combinaison stratégie-de-routage x région-de-dépôt. Nous fixons une graine aléatoire et ajoutons une gigue négligeable pour que les moyennes paraissent réalistes tout en préservant la structure équilibrée 3x3. Le poids `cell_n` de 1 dans chaque cellule indique à GLMPOWER que le plan est équilibré.

In [1]:
/* ================================================================
   Générer le jeu de données exemplaire des retards moyens
   conjecturés.
   Une ligne par cellule stratégie-de-routage x région-de-dépôt.
   ================================================================ */
DONNÉES routing_trial;
   APPELER streaminit(20260531);
   LONGUEUR routing_strategy $8 depot_region $9;
   TABLEAU strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   TABLEAU region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   TABLEAU smean[3]     _temporary_ (18.0 14.5 11.0);   /* moyennes de stratégie conjecturées */
   TABLEAU radj[3]      _temporary_ (1.5  0.0 -1.0);    /* ajustements régionaux (min)  */
   FAIRE i = 1 JUSQU_À 3;
      FAIRE j = 1 JUSQU_À 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         SORTIE;
      FIN;
   FIN;
   SUPPRIMER i j;
   ÉTIQUETTE routing_strategy="Stratégie de routage"
         depot_region="Région de dépôt"
         mean_delay_min="Retard moyen conjecturé (min)"
         cell_n="Poids de la cellule";
EXÉCUTER;

PROCÉDURE IMPRIMER DONNÉES=routing_trial ÉTIQUETTE noobs;
   VAR routing_strategy depot_region mean_delay_min cell_n;
   TITRE "Moyennes de cellules exemplaires : retard de livraison conjecturé (minutes)";
EXÉCUTER;

                      Moyennes de cellules exemplaires : retard de livraison conjecturé (minutes)                       

 Stratégie de routage     Région de dépôt   Retard moyen conjecturé (min)  Poids de la cellule
Static                 Northeast                             19.687507651                    1
Static                 Midwest                              17.9871067398                    1
Static                 West                                 16.8240210086                    1
Dynamic                Northeast                            15.9537535365                    1
Dynamic                Midwest                              14.4415169858                    1
Dynamic                West                                 12.8636389493                    1
AIOpt                  Northeast                            12.6143811724                    1
AIOpt                  Midwest                               10.893885771                    1
AIOpt                  


NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. Taille d'échantillon pour le plan global

Le plan en main, nous demandons à GLMPOWER de **résoudre la taille totale d'échantillon** (`NTOTAL = .`) à 80 % et 90 % de puissance. L'instruction `MODEL` spécifie un plan à deux facteurs avec interaction (`routing_strategy*depot_region`), `WEIGHT cell_n` déclare les poids de profil équilibrés, et `STDDEV = 8` est l'erreur quadratique moyenne supposée du retard de livraison. `NFRACTIONAL` permet à la procédure de rapporter des effectifs fractionnaires exacts avant l'arrondi supérieur.

Nous préenregistrons également trois comparaisons `CONTRAST` planifiées -- AI-Opt vs. Static, Dynamic vs. Static, et tout-réacheminement vs. Static -- qui documentent les hypothèses opérationnellement significatives que l'essai est conçu pour tester.

In [2]:
/* ================================================================
   Résoudre le nombre total de chauffeurs nécessaires pour
   atteindre 80 % et 90 % de puissance pour les effets stratégie-
   de-routage et région-de-dépôt.
   ================================================================ */
PROCÉDURE glmpower DONNÉES=routing_trial;
   CLASSE routing_strategy depot_region;
   MODÈLE mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   POIDS cell_n;
   CONTRASTE "AI-Opt vs. Static"               routing_strategy -1  0  1;
   CONTRASTE "Dynamic vs. Static"              routing_strategy -1  1  0;
   CONTRASTE "Tout réacheminement vs. Static"  routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   TITRE "Taille d'échantillon pour détecter les effets de stratégie de routage sur le retard";
   ÉTIQUETTE routing_strategy="Stratégie de routage" depot_region="Région de dépôt"
         mean_delay_min="Retard moyen (min)";
EXÉCUTER;

                      Moyennes de cellules exemplaires : retard de livraison conjecturé (minutes)                       

The GLMPOWER Procedure


                  Fixed Scenario Elements                   

Item                Value                                   
------------------  ----------------------------------------
Dependent Variable  Retard moyen (min)                      
Source              Stratégie de routage                    
Source              Région de dépôt                         
Source              Stratégie de routage*Région de dépôt    

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. Sensibilité de la puissance à la variabilité et à la taille de l'essai

La réponse sur la taille d'échantillon dépend de l'écart-type d'erreur supposé, rarement connu précisément à l'avance. Ici, nous inversons la question : nous **fixons** plusieurs effectifs totaux candidats (`NTOTAL = 45 90 180`) et **résolvons la puissance atteinte** (`POWER = .`) sur une grille d'écarts-types de retard plausibles (6, 8 et 10 minutes). Le tableau résultant est une carte de sensibilité -- il montre à quel point chaque plan d'effectif est robuste si la variabilité réelle des routes s'avère plus élevée qu'espéré.

In [3]:
/* ================================================================
   Grille de sensibilité : puissance atteinte selon les tailles
   d'essai candidates et les écarts-types de retard plausibles.
   ================================================================ */
PROCÉDURE glmpower DONNÉES=routing_trial;
   CLASSE routing_strategy depot_region;
   MODÈLE mean_delay_min = routing_strategy depot_region;
   POIDS cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   TITRE "Puissance atteinte selon les scénarios de variabilité et de taille d'essai";
   ÉTIQUETTE routing_strategy="Stratégie de routage" depot_region="Région de dépôt"
         mean_delay_min="Retard moyen (min)";
EXÉCUTER;

                      Moyennes de cellules exemplaires : retard de livraison conjecturé (minutes)                       

The GLMPOWER Procedure


         Fixed Scenario Elements         

Item                Value                
------------------  ---------------------
Dependent Variable  Retard moyen (min)   
Source              Stratégie de routage 
Source              Région de dépôt      

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. Courbe de puissance pour la décision d'effectif

Enfin, nous traçons une **courbe de puissance** -- la puissance atteinte à mesure que l'effectif total croît de 30 à 270 chauffeurs par pas de 30 -- pour le modèle stratégie-plus-région à l'écart-type attendu de 8 minutes. Résoudre `POWER = .` sur cette grille d'effectifs produit la courbe sous forme d'une série tabulée `N Total` vs. `Power`, ce qui permet de lire l'effectif où chacune des cibles conventionnelles de 80 % et 90 % est franchie.

In [4]:
/* ================================================================
   Courbe de puissance : puissance atteinte vs. total de
   chauffeurs inscrits, balayée de 30 à 270 par pas de 30 à
   l'écart-type attendu de 8 min.
   ================================================================ */
PROCÉDURE glmpower DONNÉES=routing_trial;
   CLASSE routing_strategy depot_region;
   MODÈLE mean_delay_min = routing_strategy depot_region;
   POIDS cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   TITRE "Courbe de puissance : puissance atteinte vs. total de chauffeurs inscrits";
   ÉTIQUETTE routing_strategy="Stratégie de routage" depot_region="Région de dépôt"
         mean_delay_min="Retard moyen (min)";
EXÉCUTER;

                      Moyennes de cellules exemplaires : retard de livraison conjecturé (minutes)                       

The GLMPOWER Procedure


         Fixed Scenario Elements         

Item                Value                
------------------  ---------------------
Dependent Variable  Retard moyen (min)   
Source              Stratégie de routage 
Source              Région de dépôt      

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. Interprétation et recommandation

L'analyse donne à l'équipe des opérations un plan d'effectif défendable :

- **Dimensionnement de référence (section 2).** En supposant un écart-type résiduel de 8 minutes, le modèle complet à deux facteurs (stratégie, région et leur interaction) atteint **80 % de puissance à 63 chauffeurs** et **90 % de puissance à 83 chauffeurs**. En arrondissant pour l'attrition, une cible d'effectif proche de **90 chauffeurs** franchit confortablement le seuil de 90 %.

- **La sensibilité compte (section 3).** La puissance est très sensible à la variabilité du retard. À 90 chauffeurs, la puissance atteinte passe d'environ 99 % (ET=6) à environ 87 % (ET=8) puis à environ 68 % (ET=10). Un pilote de 45 chauffeurs n'est adéquat que si la variabilité est faible (~81 % de puissance à ET=6) et est nettement sous-dimensionné (~37 %) si l'ET atteint 10. L'implication pratique : investir dans des routes cohérentes et bien instrumentées pour contenir la variabilité a autant de valeur que d'ajouter des chauffeurs.

- **La courbe de puissance (section 4).** Tracée pour le modèle stratégie-plus-région (sans terme d'interaction, l'optique utilisée pour le balayage de sensibilité), la puissance atteinte grimpe de 0.37 à 30 chauffeurs à 0.69 à 60, 0.87 à 90 et 0.95 à 120, se stabilisant au-dessus de 0.99 à partir de 180. En lisant la courbe par rapport aux cibles, 80 % de puissance arrive près de 77 chauffeurs et 90 % près de 99 -- un peu plus haut que le dimensionnement du modèle complet de la section 2 car retirer le terme d'interaction répartit le même effet sur moins de degrés de liberté du modèle.

**Recommandation :** Inscrire environ **90 chauffeurs** (30 par stratégie de routage, équilibrés sur les trois régions de dépôt). Cela franchit 90 % de puissance pour le modèle complet sous l'écart-type attendu de 8 minutes, maintient environ 87 % de puissance même sur la courbe du modèle réduit plus conservatrice, et garde l'essai assez petit pour s'exécuter en un seul trimestre d'exploitation.

*Remarque :* GLMPOWER analyse le *plan* conjecturé, pas les résultats observés -- donc la crédibilité de ces chiffres repose sur les moyennes et l'écart-type conjecturés. Les équipes devraient revoir le dimensionnement une fois qu'un bref pilote fournit une estimation empirique de la variabilité du retard de livraison.